# Basic API

A minimal end-to-end workflow: prepare pseudo-observations, fit one model,
inspect the typed result, sample, predict, and persist the fitted object.
Specialised diagnostics and alternative estimators are covered in the next
notebooks.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from pyscarcopula._utils import pobs
from pyscarcopula import GumbelCopula
from pyscarcopula.api import fit, predict, sample

## Data and fit configuration

In [2]:
TICKERS = ["BTC-USD", "ETH-USD"]
FIT = {"method": "mle"}
SEED = 2026

DATA_DIR = Path("data") if Path("data").exists() else Path("..") / "data"
prices = pd.read_csv(DATA_DIR / "crypto_prices.csv", index_col=0, sep=";")
returns = np.log(prices[TICKERS] / prices[TICKERS].shift(1)).dropna()
u = pobs(returns.to_numpy())
print(f"T={len(u)}, d={u.shape[1]}")

T=1460, d=2


## Stateless fit and typed result

In [3]:
copula = GumbelCopula(rotate=180)
result = fit(copula, u, **FIT)

print(result)
print("parameter:", result.copula_param)
pd.Series(result.diagnostics, name="value")

         copula: Gumbel copula
         method: MLE
 log_likelihood: 955.627508
   copula_param: 2.831761
        success: True
           nfev: 5
        message: CONVERGENCE: NORM OF PROJECTED GRADIENT <= PGTOL
parameter: 2.831761089780088


model_score           not_applicable
optimizer_gradient        analytical
gradient_kind             analytical
setup_derivative      not_applicable
filter_derivative     not_applicable
parameter_gradient        analytical
Name: value, dtype: str

## Sample versus predict

`sample` reproduces the fitted model. `predict` uses the fitted state and
observed history to produce the requested predictive horizon.

In [4]:
sample_u = sample(
    copula, u, result, n=2_000, rng=np.random.default_rng(SEED)
)
predicted_u = predict(
    copula, u, result, n=2_000, rng=np.random.default_rng(SEED + 1)
)

print("sample:", sample_u.shape, sample_u.mean(axis=0))
print("predict:", predicted_u.shape, predicted_u.mean(axis=0))

sample: (2000, 2) [0.48387345 0.48360044]
predict: (2000, 2) [0.50394188 0.49948132]


## Stateful convenience API and persistence

In [5]:
model = GumbelCopula(rotate=180)
model.fit(u, **FIT)

path = Path("basic_api_gumbel.json")
try:
    model.save(path)
    restored = GumbelCopula.load(path)
    restored_sample = restored.predict(
        10, rng=np.random.default_rng(SEED)
    )
finally:
    path.unlink(missing_ok=True)

print(type(restored).__name__, restored_sample.shape)

GumbelCopula (10, 2)
